In [1]:
import pandas as pd 
import numpy as np  
import joblib

In [3]:
index_names = ['unit_number', 'time_in_cycles']
setting_names = ['op_setting_1', 'op_setting_2', 'op_setting_3']
sensor_names = [f'sensor_{i}' for i in range(1, 22)]
col_names = index_names + setting_names + sensor_names

df = pd.read_csv("../CMAPSSData/test_FD001.txt", sep=r"\s+", header=None, names=col_names)

df.head()

,unit_number,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130


In [15]:
const_columns = ['sensor_1', 'sensor_5', 'sensor_6', 'sensor_10', 'sensor_16', 'sensor_18',
                 'sensor_19']
df_reduced = df.drop(columns = const_columns)

df_reduced_1 = df_reduced.drop(columns = setting_names)

df_reduced_1['compsite_id'] = df_reduced_1['unit_number']*1000 + df_reduced_1['time_in_cycles']

df_final = df_reduced_1.drop(columns = index_names)

In [16]:
df_final = df_final[[df_final.columns[-1]] + df_final.columns[:-1].tolist()]
df_final.shape

(13096, 15)

In [17]:
df_final.head()

,compsite_id,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21
0,1001,643.02,1585.29,1398.21,553.90,2388.04,9050.17,47.20,521.72,2388.03,8125.55,8.4052,392,38.86,23.3735
1,1002,641.71,1588.45,1395.42,554.85,2388.01,9054.42,47.50,522.16,2388.06,8139.62,8.3803,393,39.02,23.3916
2,1003,642.46,1586.94,1401.34,554.11,2388.05,9056.96,47.50,521.97,2388.03,8130.10,8.4441,393,39.08,23.4166
3,1004,642.44,1584.12,1406.42,554.07,2388.03,9045.29,47.28,521.38,2388.05,8132.90,8.3917,391,39.00,23.3737
4,1005,642.51,1587.19,1401.92,554.16,2388.01,9044.55,47.31,522.15,2388.03,8129.54,8.4031,390,38.99,23.4130


In [33]:
# print(df_final.columns[1:])
identifier_col = df_final.columns[0]
feature_col = df_final.columns[1:]

metadata = df_final[identifier_col]
features = df_final[feature_col]

scaler = joblib.load("../Data/scaler.pkl")

scaled_features = scaler.transform(features)

/Users/shivaram/telemetry-anomaly-detector/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


array([[0.54518072, 0.31066056, 0.26941256, ..., 0.33333333, 0.55813953,
        0.66183375],
       [0.15060241, 0.3795509 , 0.222316  , ..., 0.41666667, 0.68217054,
        0.68682684],
       [0.37650602, 0.34663179, 0.32224848, ..., 0.41666667, 0.72868217,
        0.72134769],
       ...,
       [0.67168675, 0.48201439, 0.41475354, ..., 0.58333333, 0.37209302,
        0.4293013 ],
       [0.61746988, 0.52212775, 0.62643484, ..., 0.58333333, 0.40310078,
        0.51877934],
       [0.52409639, 0.66666667, 0.72147198, ..., 0.66666667, 0.43410853,
        0.40223695]], shape=(13096, 14))

In [34]:
metadata.to_csv("../Data/Test_Data/metadata.csv")
np.save("../Data/Test_Data/scaled_features", scaled_features)

print("saved successfully")

saved successfully


In [38]:
composite_ids = metadata.astype(np.uint32).values
np.save("../Data/Test_Data/identifiers", composite_ids)
print()

AttributeError: 'numpy.ndarray' object has no attribute 'type'